In [0]:

from time import perf_counter

tables = [
    "workspace.healthcare_analytics.patients",
    "workspace.healthcare_analytics.encounters",
    "workspace.healthcare_analytics.bronze_patient_events"
]

print("HEALTHCARE TABLE PERFORMANCE CHECK")

for table in tables:
    start = perf_counter()

    count = spark.table(table).count()

    elapsed = round(perf_counter() - start, 2)

    print(f"{table}")
    print(f"Records: {count}")
    print(f"Execution time: {elapsed} seconds")
    print("-" * 50)

print("SUCCESS: Healthcare source tables verified")


HEALTHCARE TABLE PERFORMANCE CHECK
workspace.healthcare_analytics.patients
Records: 5
Execution time: 3.94 seconds
--------------------------------------------------
workspace.healthcare_analytics.encounters
Records: 7
Execution time: 0.68 seconds
--------------------------------------------------
workspace.healthcare_analytics.bronze_patient_events
Records: 5
Execution time: 0.8 seconds
--------------------------------------------------
SUCCESS: Healthcare source tables verified


In [0]:

from pyspark.sql import functions as F
from time import perf_counter

patients = spark.table(
    "workspace.healthcare_analytics.patients"
)

encounters = spark.table(
    "workspace.healthcare_analytics.encounters"
)

diagnostic_table = (
    "workspace.healthcare_analytics."
    "workflow_performance_diagnostic"
)

print("HEALTHCARE WORKFLOW PERFORMANCE TEST")

# 1. Test broadcast join and aggregation
start = perf_counter()

metrics = (
    encounters.join(
        F.broadcast(
            patients.select("patient_id", "city")
        ),
        "patient_id",
        "inner"
    )
    .groupBy("patient_id", "city")
    .agg(
        F.count("encounter_id").alias("total_encounters"),
        F.sum("treatment_cost").alias("total_treatment_cost")
    )
)

rows = metrics.collect()
join_time = round(perf_counter() - start, 2)

assert len(rows) == 5
assert sum(row["total_encounters"] for row in rows) == 7
assert round(
    sum(row["total_treatment_cost"] for row in rows), 2
) == 21800.00

print(f"PASS: Broadcast join - {join_time} seconds")

# 2. Test Delta write
start = perf_counter()

spark.createDataFrame(rows, metrics.schema) \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(diagnostic_table)

write_time = round(perf_counter() - start, 2)

print(f"PASS: Delta write - {write_time} seconds")

# 3. Verify saved records
start = perf_counter()

count = spark.table(diagnostic_table).count()

read_time = round(perf_counter() - start, 2)

assert count == 5

print(f"PASS: Delta verification - {read_time} seconds")

print("\nSUCCESS: WORKFLOW PERFORMANCE TEST COMPLETED")
print("Broadcast join:", join_time, "seconds")
print("Delta write:", write_time, "seconds")
print("Delta verification:", read_time, "seconds")


HEALTHCARE WORKFLOW PERFORMANCE TEST
PASS: Broadcast join - 4.07 seconds
PASS: Delta write - 9.85 seconds
PASS: Delta verification - 0.58 seconds

SUCCESS: WORKFLOW PERFORMANCE TEST COMPLETED
Broadcast join: 4.07 seconds
Delta write: 9.85 seconds
Delta verification: 0.58 seconds
